# Week 5: Trial of Reflection
A single layer fails on the XOR-like trial dataset. A ReLU hidden layer lets the network learn combinations of tells.

In [1]:
from pathlib import Path
import sys
import numpy as np

# Works when opened from the repo root or the week5 folder.
folder = Path.cwd() if Path("part1_single_layer_fails.py").exists() else Path.cwd() / "week5"
sys.path.insert(0, str(folder))
from part1_single_layer_fails import tells, strike, single_layer_train
from part2_forward_hidden import relu, forward
from part3_one_backprop_step import one_step
from part4_full_training_loop import train


## Part 1: Single-layer failure
Foot and guard each appear in both strikes and holds. Exhale is always 1, so it acts as a constant input. A weighted sum cannot make exactly one of foot and guard mean strike while both or neither mean hold.

In [2]:
single_layer_train(tells, strike.ravel(), 0.1, 60, 1)

epoch: 10 error: 1.5858923703029502
epoch: 20 error: 1.3411393242635645
epoch: 30 error: 1.2724452140939488
epoch: 40 error: 1.249877829551867
epoch: 50 error: 1.2413639273585795
epoch: 60 error: 1.2377688703690684
Final Weights: [ 0.00937726 -0.04602622  0.43220539]
Sensing 0: Prediction = 0.4415826451369658, Goal = 1
Sensing 1: Prediction = 0.38617916667439367, Goal = 1
Sensing 2: Prediction = 0.4322053850644371, Goal = 0
Sensing 3: Prediction = 0.3955564267469224, Goal = 0


## Part 2: Forward pass
The shapes are (1, 3) @ (3, 4) = (1, 4), then (1, 4) @ (4, 1) = (1, 1). ReLU removes negative hidden activations; the output has no activation.

In [3]:
np.random.seed(1)
w01 = 2 * np.random.random((3, 4)) - 1
w12 = 2 * np.random.random((4, 1)) - 1
print("Weight shapes:", w01.shape, w12.shape)
for i in range(4):
    x = tells[i:i+1]
    hidden, prediction = forward(x, w01, w12)
    print(i, "shapes:", x.shape, hidden.shape, prediction.shape)
    print("hidden:", hidden, "prediction:", prediction)


Weight shapes: (3, 4) (4, 1)
0 shapes: (1, 3) (1, 4) (1, 1)
hidden: [[0.         0.51828245 0.         0.        ]] prediction: [[0.39194327]]
1 shapes: (1, 3) (1, 4) (1, 1)
hidden: [[0.         0.         0.         0.06156045]] prediction: [[0.02098811]]
2 shapes: (1, 3) (1, 4) (1, 1)
hidden: [[0.         0.07763347 0.         0.370439  ]] prediction: [[0.18500476]]
3 shapes: (1, 3) (1, 4) (1, 1)
hidden: [[0. 0. 0. 0.]] prediction: [[0.]]


## Part 3: One backprop step
The output delta (1, 1) multiplies the transposed output weights (1, 4) to give a hidden delta (1, 4). The ReLU derivative blocks gradients for inactive units. We run another forward pass with the updated weights to measure the new squared error.

In [4]:
x, target = tells[0:1], strike[0:1]
hidden, before = forward(x, w01, w12)
new01, new12, _, _ = one_step(x, target, w01, w12, 0.2)
_, after = forward(x, new01, new12)
print("Prediction before/after:", before, after)
print("Squared error before/after:", (before-target)**2, (after-target)**2)
print("Weight shapes before:", w01.shape, w12.shape)
print("Weight shapes after:", new01.shape, new12.shape)
print("Weight [1, 0] check:", w12[1, 0] - 0.2 * hidden[0, 1] * (before-target).item())
print("Actual updated weight:", new12[1, 0])


Prediction before/after: [[0.39194327]] [[0.57530017]]
Squared error before/after: [[0.36973299]] [[0.18036995]]
Weight shapes before: (3, 4) (4, 1)
Weight shapes after: (3, 4) (4, 1)
Weight [1, 0] check: 0.8192639001087971
Actual updated weight: 0.8192639001087971


## Part 4: Full training
One update per sensing is repeated for 60 epochs. The hidden layer learns two useful detectors, allowing the output to fit the pattern.

In [5]:
w01, w12, errors = train(tells, strike, 0.2, 60, 4, 1)
hidden, predictions = forward(tells, w01, w12)
print("Predictions and goals:")
print(np.column_stack((predictions, strike)))
print("Input weights:", w01.round(2), sep="\n")
print("Output weights:", w12.round(2), sep="\n")
print("Hidden activations:", hidden.round(3), sep="\n")


Epoch 1: total error = 1.414206
Epoch 10: total error = 0.634231
Epoch 20: total error = 0.358384
Epoch 30: total error = 0.083018
Epoch 40: total error = 0.006467
Epoch 50: total error = 0.000329
Epoch 60: total error = 0.000015
Predictions and goals:
[[1.         1.        ]
 [0.99855326 1.        ]
 [0.00263144 0.        ]
 [0.         0.        ]]
Input weights:
[[-0.17  0.91 -1.   -0.9 ]
 [-0.71 -0.93 -0.63  0.9 ]
 [-0.21 -0.03 -0.16  0.  ]]
Output weights:
[[-0.59]
 [ 1.14]
 [-0.95]
 [ 1.11]]
Hidden activations:
[[0.    0.877 0.    0.   ]
 [0.    0.    0.    0.899]
 [0.    0.    0.    0.002]
 [0.    0.    0.    0.   ]]


Units 0 and 2 have negative incoming weights and stay inactive on all four inputs. Unit 1 has foot weight 0.91 and guard weight -0.93, so it detects foot without guard. Unit 3 has foot weight -0.90 and guard weight 0.90, so it detects guard without foot; its exhale weight is small, not exactly zero.

In [6]:
import contextlib
import io
print("Hidden size | seed 1 | seed 2 | seed 3")
for size in [1, 2, 4, 8, 16]:
    results = []
    for seed in [1, 2, 3]:
        with contextlib.redirect_stdout(io.StringIO()):
            _, _, history = train(tells, strike, 0.2, 60, size, seed)
        results.append(f"{history[-1]:.6f}")
    print(size, " | ".join(results))


Hidden size | seed 1 | seed 2 | seed 3
1 2.000000 | 2.000095 | 2.000004
2 2.000000 | 2.000000 | 0.066403
4 0.000015 | 1.000000 | 1.000000
8 0.000000 | 0.033719 | 0.000000
16 0.000000 | 0.000000 | 0.000000


Using final epoch error < 0.01, sizes 1 and 2 fail all three seeds, sizes 4 and 8 have mixed results, and size 16 succeeds all three times. The smallest tested success on seed 1 is 4, while the smallest meeting this threshold across all three seeds is 16, because initialization can leave too few useful ReLU units.

## Part 5: Tests
Eight tests check activations, single-layer failure, forward shapes, one-step improvement, convergence, and repeatability.

In [7]:
import subprocess
result = subprocess.run([sys.executable, str(folder / "test_trial.py")], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    raise RuntimeError(result.stderr or "Tests failed")


PASS: test_hidden_shape
PASS: test_one_step_reduces_error
PASS: test_output_shape
PASS: test_relu
PASS: test_relu_derivative
PASS: test_single_layer_fails
PASS: test_training_converges
PASS: test_training_is_deterministic
8/8 tests passed

